In [ ]:
!pip install langchain langchain-community chromadb sentence-transformers groq ragas datasets langgraph -q
from langchain_community.vectorstores import Chroma

from google.colab import userdata
from groq import Groq

import os

In [ ]:
!pip install wikipedia-api -q

import wikipediaapi
import os

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rag-project/1.0'
)

os.makedirs("wiki_docs", exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 8.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls wiki_docs

In [ ]:
!cp -r wiki_docs /content/drive/MyDrive/

In [ ]:
print(len(os.listdir("wiki_docs")))

0


In [ ]:
import wikipediaapi
import os

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rag-project/1.0'
)

os.makedirs("wiki_docs", exist_ok=True)

topics = [
    "Machine learning",
    "Deep learning",
    "Artificial neural network",
    "Convolutional neural network",
    "Recurrent neural network",
    "Long short-term memory",
    "Transformer (machine learning model)",
    "Large language model",
    "Backpropagation",
    "Gradient descent",
    "Computer vision",
    "Image classification",
    "Object detection",
    "Support vector machine",
    "Random forest",
    "Decision tree learning",
    "XGBoost",
    "K-nearest neighbors algorithm"
]

for topic in topics:
    page = wiki.page(topic)

    if page.exists():
        filename = topic.replace(" ", "_").replace("/", "_") + ".txt"

        with open(f"wiki_docs/{filename}", "w", encoding="utf-8") as f:
            f.write(page.text)

        print(f"Saved: {filename}")
    else:
        print(f"Not found: {topic}")

Saved: Machine_learning.txt
Saved: Deep_learning.txt
Saved: Artificial_neural_network.txt
Saved: Convolutional_neural_network.txt
Saved: Recurrent_neural_network.txt
Saved: Long_short-term_memory.txt
Saved: Transformer_(machine_learning_model).txt
Saved: Large_language_model.txt
Saved: Backpropagation.txt
Saved: Gradient_descent.txt
Saved: Computer_vision.txt
Saved: Image_classification.txt
Saved: Object_detection.txt
Saved: Support_vector_machine.txt
Saved: Random_forest.txt
Saved: Decision_tree_learning.txt
Saved: XGBoost.txt
Saved: K-nearest_neighbors_algorithm.txt


In [ ]:
import os

files = os.listdir("wiki_docs")

print("Number of files:", len(files))
print(files)

Number of files: 18
['Computer_vision.txt', 'Large_language_model.txt', 'Object_detection.txt', 'XGBoost.txt', 'Random_forest.txt', 'Support_vector_machine.txt', 'Long_short-term_memory.txt', 'Recurrent_neural_network.txt', 'Machine_learning.txt', 'Transformer_(machine_learning_model).txt', 'Image_classification.txt', 'Convolutional_neural_network.txt', 'Artificial_neural_network.txt', 'Deep_learning.txt', 'Gradient_descent.txt', 'Decision_tree_learning.txt', 'K-nearest_neighbors_algorithm.txt', 'Backpropagation.txt']


In [ ]:
with open("wiki_docs/Machine_learning.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:1000])

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.
From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation under this framework.

History
The term machine learning was coined in 1959 by Arthur Samuel, an IBM

In [ ]:
!pip show langchain

Name: langchain
Version: 1.3.6
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: ragas


In [ ]:
!pip install langchain-community -q

In [ ]:
# Install Required Libraries,Import
!pip install langchain chromadb langchain-google-genai -q
!pip install langchain-text-splitters -q
import os

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.5 MB/s eta 0:00:00


In [ ]:
import os
from langchain_core.documents import Document

documents = []

for filename in os.listdir("wiki_docs"):
    filepath = os.path.join("wiki_docs", filename)

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append(
        Document(
            page_content=text,
            metadata={"source": filename}
        )
    )

print("Documents Loaded:", len(documents))

Documents Loaded: 18


In [ ]:
print(documents[0].metadata)

print("\nFirst 500 characters:\n")

print(documents[0].page_content[:500])

{'source': 'Computer_vision.txt'}

First 500 characters:

Computer vision tasks include methods for acquiring, processing, analyzing, and understanding digital images, and extraction of high-dimensional data from the real world in order to produce numerical or symbolic information, e.g. in the form of decisions. "Understanding" in this context signifies the transformation of visual images into descriptions of the world that make sense to thought processes and can elicit appropriate action. This image understanding can be seen as the disentangling of sy


**Splitting into chunks of 500 words each**

In [ ]:
total_words = sum(len(doc.page_content.split()) for doc in documents)

print("Total words:", total_words)

Total words: 103803


In [ ]:
from langchain_core.documents import Document

word_chunks = []

for doc in documents:
    words = doc.page_content.split()

    chunk_size = 500
    overlap = 50

    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk_text = " ".join(words[start:end])

        word_chunks.append(
            Document(
                page_content=chunk_text,
                metadata=doc.metadata
            )
        )

        start += chunk_size - overlap

print("Total Chunks:", len(word_chunks))

Total Chunks: 239


In [ ]:
word_counts = [len(chunk.page_content.split()) for chunk in word_chunks]

print("Total Chunks:", len(word_chunks))
print("Average words:", sum(word_counts)/len(word_counts))
print("Min words:", min(word_counts))
print("Max words:", max(word_counts))

Total Chunks: 239
Average words: 480.4267782426778
Min words: 30
Max words: 500


In [ ]:
print(len(word_chunks[0].page_content.split()))

500


**Embedding Starting**

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [ ]:
sample_embedding = embeddings.embed_query(
    word_chunks[0].page_content
)

print("Embedding dimension:", len(sample_embedding))

Embedding dimension: 3072


In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
test_chunks = word_chunks[:10]

vectorstore = Chroma.from_documents(
    documents=test_chunks,
    embedding=embeddings,
    persist_directory="./test_db"
)

print("Test successful!")

Test successful!


**OPTION 2 - USING a local embedding model**

---



In [ ]:
!pip install sentence-transformers chromadb langchain-community -q
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_2802/1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = Chroma.from_documents(
    documents=word_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Done!")
print("Total indexed:", vectorstore._collection.count())

Done!
Total indexed: 239


In [ ]:
# TO LOAD EXISTING CHROMADB
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

**Initially, Gemini embeddings were used for generating vector representations of document chunks. However, during bulk ingestion of large chunks into ChromaDB, repeated API quota and rate-limit issues were encountered. To ensure smooth and scalable indexing, the embedding model was switched to the local Sentence-Transformers model `all-MiniLM-L6-v2`.**

This model generates embeddings locally inside the Colab environment without relying on external API calls, which avoids quota limitations and enables faster large-scale document processing. The generated embeddings are then stored in ChromaDB, where semantic similarity search can be performed efficiently during retrieval in the RAG pipeline.

**Trying with a specific query**

In [ ]:
query = "What is backpropagation?"

results = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}\n")
    print(doc.page_content[:500])


Result 1

In machine learning, backpropagation is a gradient computation method commonly used for training a neural network in computing parameter updates. It is an efficient application of the chain rule to neural networks. Backpropagation efficiently computes the gradient of the loss with respect to the network weights for a single input–output example. It does this by propagating derivatives backward, one layer at a time, from the output layer to the input layer, thereby avoiding redundant chain-rule c

Result 2

classify non-linearily separable pattern classes. Subsequent developments in hardware and hyperparameter tunings have made end-to-end stochastic gradient descent the currently dominant training technique. In 1969, Kunihiko Fukushima introduced the ReLU (rectified linear unit) activation function. The rectifier has become the most popular activation function for deep learning. Deep learning architectures for convolutional neural networks (CNNs) with convolutional layers and

**Adding Dynamic user input**

In [ ]:
!pip install groq
from google.colab import userdata
from groq import Groq
import os

try:
    # Load GROQ API KEY
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_KEY")
    # Create Groq Client
    client = Groq(
        api_key=os.environ["GROQ_API_KEY"]
    )

    # Take User Query
    query = input("Enter your question: ")

    # Empty query check
    if not query.strip():
        raise ValueError("Query cannot be empty.")

    # Retrieve Top-5 Similar Chunks
    results = vectorstore.similarity_search_with_score(
        query,
        k=5
    )

    # Filter Relevant Chunks
    relevant_docs = []

    print("\nTop Retrieved Chunks:\n")

    for i, (doc, score) in enumerate(results, start=1):

        # Lower score = better similarity
        if score < 1.0:

            relevant_docs.append(doc)

            print(f"\nResult {i}")
            print(f"Similarity Score: {score}\n")

            print(doc.page_content[:500])

    # Check if Relevant Docs Exist
    if not relevant_docs:
        raise ValueError("No sufficiently relevant documents found.")

    # Create Context
    context = "\n\n".join(
        [doc.page_content for doc in relevant_docs]
    )

    # Create Prompt
    prompt = f"""
    Answer the question using ONLY the context below.

    If the answer is not present in the context,
    clearly say that the information is unavailable.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # Generate Response from Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    # Print Final Answer
    print("\nFinal Answer:\n")
    print(response.choices[0].message.content)

# Exception Handling
except Exception as e:

    print("\nError occurred:")
    print(e)

Enter your question: Is ML better than AI

Top Retrieved Chunks:


Result 1
Similarity Score: 0.947452187538147

the machine learning field: "A computer program is said to learn from experience E with respect to some class of tasks T and performance measure P if its performance at tasks in T, as measured by P, improves with experience E." This definition of the tasks in which machine learning is concerned is fundamentally operational rather than defining the field in cognitive terms. This follows Alan Turing's proposal in his paper "Computing Machinery and Intelligence", in which the question, "Can machine

Final Answer:

The information is unavailable. The context provides a historical and technical overview of the machine learning field and its relationship to artificial intelligence, but it does not make a direct comparison or judgment about which one is "better".


**LangGraph Workflow Pipeline**

In [ ]:
!pip install langgraph -q
from typing import TypedDict
from langgraph.graph import StateGraph, END

In [ ]:
# Define Shared state
class RAGState(TypedDict):

    query: str
    retrieved_docs: list
    context: str
    answer: str

In [ ]:
# Retrieval Node
def retrieve(state):

    try:

        query = state["query"]

        results = vectorstore.similarity_search_with_score(
            query,
            k=5
        )

        relevant_docs = []

        for doc, score in results:

            if score < 1.0:
                relevant_docs.append(doc)

        if not relevant_docs:
            raise ValueError("No relevant documents found.")

        return {
            "retrieved_docs": relevant_docs
        }

    except Exception as e:

        print("Retrieval Error:", e)

        return {
            "retrieved_docs": []
        }

In [ ]:
# Context Builder Node

def build_context(state):

    try:

        docs = state["retrieved_docs"]

        context = "\n\n".join(
            [doc.page_content for doc in docs]
        )

        return {
            "context": context
        }

    except Exception as e:

        print("Context Error:", e)

        return {
            "context": ""
        }

In [ ]:
# Generation Node
def generate_answer(state):

    try:

        query = state["query"]
        context = state["context"]

        prompt = f"""
        Answer the question using ONLY the context below.

        If the answer is unavailable in the context,
        clearly mention that.

        Context:
        {context}

        Question:
        {query}

        Answer:
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        answer = response.choices[0].message.content

        return {
            "answer": answer
        }

    except Exception as e:

        print("Generation Error:", e)

        return {
            "answer": "Error generating answer."
        }

In [ ]:
# Create Graph
graph = StateGraph(RAGState)

# Add Nodes
graph.add_node("retrieve", retrieve)

graph.add_node("build_context", build_context)

graph.add_node("generate_answer", generate_answer)

# Define Flow
graph.set_entry_point("retrieve")

graph.add_edge("retrieve", "build_context")

graph.add_edge("build_context", "generate_answer")

graph.add_edge("generate_answer", END)

# Compile Graph
rag_graph = graph.compile()

print("LangGraph pipeline created successfully!")

LangGraph pipeline created successfully!


In [ ]:
query = input("Enter your question: ")

result = rag_graph.invoke(
    {
        "query": query
    }
)

print("\nFinal Answer:\n")

print(result["answer"])

Enter your question: any algorithm in ML

Final Answer:

Empirical Risk Minimization


**RAGAS EVALUATION**

In [ ]:
!pip install ragas datasets -q
from datasets import Dataset

In [ ]:
questions = [

    "Why are CNNs considered more effective for image processing tasks compared to traditional neural networks?",

    "How does backpropagation help a neural network improve its predictions over time?",

    "Why is gradient descent important during the training of machine learning models?",

    "What problem does LSTM solve that traditional recurrent neural networks struggle with?",

    "How do convolutional layers help CNNs identify patterns in images?",

    "Why is the vanishing gradient problem a major issue in deep neural networks?",

    "How are transformers different from CNNs in handling data and learning patterns?",

    "Why do neural networks require activation functions instead of simple linear operations?",

    "How does semantic similarity search improve information retrieval in RAG systems?",

    "Why can overfitting reduce the real-world performance of a machine learning model?",

    "How does chunking affect the performance and accuracy of a RAG pipeline?",

    "Why are embeddings important in vector databases such as ChromaDB?",

    "How does a CNN learn low-level and high-level image features during training?",

    "Why is context retrieval necessary before generating answers in a RAG system?",

    "How do attention mechanisms improve the performance of large language models?",

    "Why are pretrained models commonly used in modern deep learning applications?",

    "How does similarity scoring help filter irrelevant chunks during retrieval?",

    "Why is exception handling important in AI pipelines and workflow systems?",

    "How does LangGraph help organize and manage a multi-step RAG workflow?",

    "Why can a RAG system sometimes produce incorrect or incomplete answers even after retrieval?"

]

In [ ]:
answers = []
contexts = []

for question in questions:

    try:

        result = rag_graph.invoke(
            {
                "query": question
            }
        )

        answers.append(result["answer"])

        retrieved = vectorstore.similarity_search(
            question,
            k=3
        )

        contexts.append(
            [doc.page_content for doc in retrieved]
        )

        print(f"Completed: {question}")

    except Exception as e:

        print(f"Error for question: {question}")

        print(e)

        answers.append("Error")

        contexts.append([])

Completed: Why are CNNs considered more effective for image processing tasks compared to traditional neural networks?
Completed: How does backpropagation help a neural network improve its predictions over time?
Completed: Why is gradient descent important during the training of machine learning models?
Completed: What problem does LSTM solve that traditional recurrent neural networks struggle with?
Completed: How do convolutional layers help CNNs identify patterns in images?
Retrieval Error: No relevant documents found.
Completed: Why is the vanishing gradient problem a major issue in deep neural networks?
Completed: How are transformers different from CNNs in handling data and learning patterns?
Completed: Why do neural networks require activation functions instead of simple linear operations?
Retrieval Error: No relevant documents found.
Completed: How does semantic similarity search improve information retrieval in RAG systems?
Retrieval Error: No relevant documents found.
Completed

In [ ]:
ground_truths = [

    "CNNs are effective for image processing because they capture spatial features using convolutional layers.",

    "Backpropagation updates neural network weights by computing gradients.",

    "Gradient descent minimizes loss by iteratively updating parameters.",

    "LSTM solves long-term dependency problems in sequence learning.",

    "Convolutional layers detect important image patterns and features.",

    "Vanishing gradients make deep neural network training difficult.",

    "Transformers use attention mechanisms while CNNs use convolutions.",

    "Activation functions help neural networks learn non-linear relationships.",

    "Semantic similarity search retrieves contextually related information.",

    "Overfitting causes poor generalization on unseen data.",

    "Chunking affects retrieval quality and context relevance in RAG systems.",

    "Embeddings convert text into vector representations for similarity search.",

    "CNNs progressively learn simple and complex image features.",

    "Context retrieval grounds answers using external information.",

    "Attention mechanisms help models focus on important information.",

    "Pretrained models reduce training time and improve performance.",

    "Similarity scoring filters less relevant retrieval results.",

    "Exception handling prevents workflow failures and improves robustness.",

    "LangGraph organizes AI workflows using graph-based execution.",

    "RAG systems may fail because of weak retrieval or incomplete context."

]

In [ ]:
dataset = Dataset.from_dict(
    {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths
    }
)

In [ ]:
evaluation_results = []

for i in range(len(questions)):

    question = questions[i]
    answer = answers[i]
    context = contexts[i]

    evaluation_prompt = f"""
    You are evaluating a RAG system.

    Evaluate the following answer based on:

    1. Faithfulness:
    Is the answer supported by the retrieved context?

    2. Answer Relevancy:
    Does the answer properly address the question?

    3. Context Quality:
    Was the retrieved context useful?

    Give scores out of 10.

    Also provide a short explanation.

    Question:
    {question}

    Retrieved Context:
    {context}

    Generated Answer:
    {answer}
    """

    try:

        response = client.chat.completions.create(

            model="llama-3.3-70b-versatile",

            messages=[
                {
                    "role": "user",
                    "content": evaluation_prompt
                }
            ]
        )

        evaluation = response.choices[0].message.content

        evaluation_results.append({

            "question": question,

            "evaluation": evaluation
        })

        print(f"\nCompleted Evaluation: {i+1}")

    except Exception as e:

        print(f"\nEvaluation Error at question {i+1}")
        print(e)


Completed Evaluation: 1

Completed Evaluation: 2

Completed Evaluation: 3

Completed Evaluation: 4

Completed Evaluation: 5

Completed Evaluation: 6

Completed Evaluation: 7

Completed Evaluation: 8

Completed Evaluation: 9

Completed Evaluation: 10

Completed Evaluation: 11

Completed Evaluation: 12

Completed Evaluation: 13

Completed Evaluation: 14

Completed Evaluation: 15

Completed Evaluation: 16

Completed Evaluation: 17

Completed Evaluation: 18

Completed Evaluation: 19

Completed Evaluation: 20


In [ ]:
for item in evaluation_results:

    print("\nQUESTION:\n")
    print(item["question"])

    print("\nEVALUATION:\n")
    print(item["evaluation"])

    print("\n" + "="*80)


QUESTION:

Why are CNNs considered more effective for image processing tasks compared to traditional neural networks?

EVALUATION:

Here are my evaluations:

1. **Faithfulness: 9/10**
The answer is supported by the retrieved context. The context explains how CNNs exploit spatially local correlation in natural images through local connectivity, shared weights, and pooling layers. The answer accurately summarizes these key points from the context.

2. **Answer Relevancy: 9/10**
The answer properly addresses the question. It explains why CNNs are more effective for image processing tasks compared to traditional neural networks, highlighting the advantages of local connectivity, shared weights, and pooling layers.

3. **Context Quality: 8/10**
The retrieved context is useful, providing a detailed explanation of CNNs, their architecture, and their advantages for image processing tasks. However, the context is quite lengthy and includes some irrelevant information, such as the history of CN